# Day 2 Project — Engineering Knowledge Assistant
Everything is on the table: heading-aware chunks, two embedders, a bounded labelled context, a
strict answer contract with validated citations, visible state, two scorecards. The project
assembles them, measures honestly, changes exactly one layer, and measures again.

### Step 1 — Assemble, deliberately from the weaker baseline

We start on the hash embedder on purpose: a fix you cannot measure is a fix you cannot defend. The
assistant object is the one from 2.5 — no new machinery, just a choice of parts.

In [ ]:
baseline = KnowledgeAssistant(hash_index, top_k=3)
print("embedder :", type(baseline.index.embedder).__name__)
print("chunks   :", len(baseline.index.chunks), "| top_k:", baseline.top_k, "| context budget:", baseline.budget, "chars")

state = baseline.answer("What should occupants do if the battery cabinet is hissing?")
print("\nstatus   :", state.status)
for item in state.retrieved:
    print(f"   rank {item.rank}  score {item.score:.3f}  {item.chunk.chunk_id:42} {item.chunk.section}")
print("abstained:", state.answer.abstained, "| grounded:", state.answer.grounded,
      "| citations:", [c.chunk_id for c in state.answer.citations])
print("answer   :", state.answer.answer[:200], "...")

### Step 2 — Both scorecards over the ten golden cases

Retrieval first, answers second, never merged into one headline number.

In [ ]:
retrieval_before = evaluate_retrieval(baseline.index, cases, top_k=baseline.top_k)
answers_before = evaluate_answers(baseline, cases)

print("RETRIEVAL")
print(render_table(retrieval_before, ["id", "answerable", "source_hit", "section_hit", "expected_rank"]))
print("  rates :", summarize(retrieval_before, ["source_hit", "section_hit"]))
print("  counts:", summarize_detail(retrieval_before, ["source_hit", "section_hit"]))

print("\nANSWERS")
print(render_table(answers_before, ["id", "answerable", "abstained", "abstention_correct",
                                    "citation_correct", "citation_provenance_ok", "essential_term_coverage"]))
print("  rates :", summarize(answers_before, ANSWER_FIELDS))
print("  counts:", summarize_detail(answers_before, ANSWER_FIELDS))
print("  terms :", summarize_essential_terms(answers_before))

### Step 3 — Diagnose, then change one layer and re-measure both halves

Name each failure and its layer first; guessing at a prompt before this step is the most common way
to waste a day. The diagnosis says *representation*, so we change the embedder and nothing else (or
widen `top_k`, if the trained model could not be loaded — the printout says which fix ran).

In [ ]:
print("FAILING CASES AND THE LAYER THAT OWNS THEM")
for record in retrieval_before:
    if record["answerable"] and not record["section_hit"]:
        case = by_id[record["id"]]
        print(f"  {record['id']}: {case.question}")
        print("     expected :", case.expected_source, "|", case.expected_section)
        print("     retrieved:", record["retrieved_ids"])
        print("     LAYER    : representation - the question paraphrases the document")
for record in answers_before:
    if not record["abstention_correct"]:
        print(f"  {record['id']}: abstained on an answerable question")
        print("     LAYER    : downstream of retrieval - the section was never supplied")

if semantic_index is not None:
    improved, fix = KnowledgeAssistant(semantic_index, top_k=3), "swapped the embedder (representation layer)"
else:
    improved, fix = KnowledgeAssistant(hash_index, top_k=5), "widened top_k to 5 (retrieval-depth layer)"
print("\nFIX APPLIED:", fix)

retrieval_after = evaluate_retrieval(improved.index, cases, top_k=improved.top_k)
answers_after = evaluate_answers(improved, cases)
print("retrieval before:", summarize(retrieval_before, ["source_hit", "section_hit"]),
      summarize_detail(retrieval_before, ["section_hit"]))
print("retrieval after :", summarize(retrieval_after, ["source_hit", "section_hit"]),
      summarize_detail(retrieval_after, ["section_hit"]))
for old, new in zip(retrieval_before, retrieval_after):
    if old["section_hit"] != new["section_hit"]:
        print(f"   {old['id']}: section_hit {old['section_hit']} -> {new['section_hit']}"
              f" (expected chunk rank {old['expected_rank']} -> {new['expected_rank']})")

print("\nanswers before  :", summarize(answers_before, ANSWER_FIELDS), summarize_essential_terms(answers_before))
print("answers after   :", summarize(answers_after, ANSWER_FIELDS), summarize_essential_terms(answers_after))
print(render_table(answers_after, ["id", "abstained", "abstention_correct", "citation_correct",
                                   "citation_provenance_ok", "essential_term_coverage"]))

### Step 4 — What is left, and which layer owns it now

A retrieval fix is only real if the answer report agrees, and the residue says where tomorrow's
work is: a case scoring zero on essential terms while its section *is* retrieved has moved from a
retrieval defect to a generation one.

In [ ]:
remaining = [r for r in answers_after if r["essential_terms_total"] and r["essential_term_coverage"] == 0]
if not remaining:
    print("Every scored case now covers at least one essential term.")
for record in remaining:
    rank = next(r["expected_rank"] for r in retrieval_after if r["id"] == record["id"])
    print(record["id"], "still covers none of its essential terms; missing:", record["missing_terms"])
    print("   expected chunk is now retrieved at rank", rank, "- retrieval is no longer the defect.")
    print("   Our mock quotes whichever retrieved chunk matches the most question words, and that is")
    print("   a GENERATION limit. A live model reads all three passages and can do better; the two")
    print("   separate reports are what made the hand-over visible.")

### Try it yourself

Add two golden cases of your own, one answerable and one not. Predict whether the assistant
abstains on the second before running the cell.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
my_cases = [
    GoldenCase(id="mine-01", question="What is recorded during the monthly visual inspection?",
               answerable=True, expected_source="battery_safety.md", expected_section="Inspection",
               essential_terms=["corrosion", "cable damage"]),
    GoldenCase(id="mine-02", question="Which supplier services the inverters?",
               answerable=False, expected_source=None, expected_section=None, essential_terms=[]),
]
print(render_table(evaluate_retrieval(improved.index, my_cases, top_k=improved.top_k),
                   ["id", "answerable", "source_hit", "section_hit", "expected_rank"]))
print()
print(render_table(evaluate_answers(improved, my_cases),
                   ["id", "abstained", "abstention_correct", "citation_correct", "essential_term_coverage"]))
for record in evaluate_answers(improved, my_cases):
    if record["id"] == "mine-02" and not record["abstained"]:
        print("\nmine-02 did NOT abstain: the corpus never names a supplier, but the word 'inverter'")
        print("appears in a retrieved passage, so the lexical mock believes it has evidence. With a")
        print("key, a real model reads the passage and abstains.")
print("\nA golden case is only a question plus what you already knew about its answer. Ten of them,")
print("kept honest, are worth more than any single impressive demo.")

### Checkpoint

**1. Your assistant answers a question wrongly. What do you inspect first, and why?**

<details><summary>Show answer</summary>

The retrieved chunks and their scores, before touching the prompt. If the expected section is missing, no wording change can help and the work belongs in chunking, the embedder or top-k.

</details>

**2. What is the difference between knowledge, context, state and memory in this project?**

<details><summary>Show answer</summary>

Knowledge is the indexed corpus, in no request. Context is the passages plus instructions inside one call, gone afterwards. State is the `KnowledgeState` we own during the run. Memory is what survives between runs: Day 3.

</details>

### Recap

- **Limitation seen:** an assistant that prints only a final answer hides which layer failed.
- **Layer added:** the assembled project plus two scorecards over ten known cases.
- **Evidence:** the golden set named the baseline's misses, one layer changed, and both scorecards moved.